<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>Lab: Tokenization and Vocabulary Construction using PyTorch</h1> 


Estimated time needed: **30** minutes


<h2>Learning Objectives</h2>After completing this lab, you will be able to:<ul><li> Implement tokenization techniques, including word-level and subword methods, using PyTorch tools</li><li> Build a vocabulary from a text corpus with frequency-based filtering</li><li> Convert tokens into numerical indices and reconstruct text from indices</li><li> Use special tokens such as padding (PAD), unknown (UNK), beginning of sequence (BOS), and end of sequence (EOS) when preparing text sequences for model input</li><li> Apply padding and batching for variable-length text inputs</li></ul> 


<h2>Table of Contents</h2>
<p>In this lab, you will tokenize text, build a vocabulary, encode and decode token IDs, and batch variable-length text inputs with PyTorch.</p>

<ul>
    <li><a href="#Create-a-Text-Corpus">Create a Text Corpus</a></li>
    <li><a href="#Word-level-Tokenization">Word-level Tokenization</a></li>
    <li><a href="#Subword-Tokenization">Subword Tokenization</a></li>
    <li><a href="#Build-a-Vocabulary-with-Frequency-Filtering">Build a Vocabulary with Frequency Filtering</a></li>
    <li><a href="#Encode-Tokens-as-Indices">Encode Tokens as Indices</a></li>
    <li><a href="#Decode-Indices-Back-to-Text">Decode Indices Back to Text</a></li>
    <li><a href="#Special-Token-Handling">Special Token Handling</a></li>
    <li><a href="#Padding-and-Batching">Padding and Batching</a></li>
</ul>

<hr>


The following required libraries are __not__ pre-installed in the Skills Network Labs environment. __You will need to run the following cells__ to install or import them:


In [1]:
# Install PyTorch

!pip install torch

In [2]:
# Import the libraries we need for this lab

import re
from collections import Counter

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

torch.manual_seed(2)

/opt/conda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))

<!--Empty Space for separating topics-->


<h2 id="Corpus">Create a Text Corpus</h2>


Create a small text corpus. Each item is a short sentence that will be tokenized and converted into model-ready numerical input:


In [3]:
# Create a text corpus

corpus = [
    "PyTorch models learn from tokenized text data.",
    "Tokenization converts raw text into useful tokens.",
    "Vocabulary construction maps tokens to numerical indices.",
    "Rare words may be replaced with unknown tokens.",
    "Padding allows variable length sentences to form batches.",
    "Subword tokens can represent prefixes, roots, and suffixes.",
]

for sentence in corpus:
    print(sentence)

PyTorch models learn from tokenized text data.
Tokenization converts raw text into useful tokens.
Vocabulary construction maps tokens to numerical indices.
Rare words may be replaced with unknown tokens.
Padding allows variable length sentences to form batches.
Subword tokens can represent prefixes, roots, and suffixes.

Check how many words appear in each sentence before tokenization rules are applied:


In [4]:
# Inspect sentence lengths

word_counts = [len(sentence.split()) for sentence in corpus]

print(word_counts)
print("Shortest sentence:", min(word_counts))
print("Longest sentence:", max(word_counts))

[7, 7, 7, 8, 8, 8]
Shortest sentence: 7
Longest sentence: 8

<!--Empty Space for separating topics-->


<h2 id="WordTokenization">Word-level Tokenization</h2>


Word-level tokenization splits text into words. The tokenizer below lowercases text and removes punctuation by keeping word characters only:


In [5]:
# Create a word-level tokenizer

def word_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

Apply the tokenizer to the corpus:


In [6]:
# Tokenize the corpus by words

word_tokens = [word_tokenize(sentence) for sentence in corpus]

for tokens in word_tokens:
    print(tokens)

['pytorch', 'models', 'learn', 'from', 'tokenized', 'text', 'data']
['tokenization', 'converts', 'raw', 'text', 'into', 'useful', 'tokens']
['vocabulary', 'construction', 'maps', 'tokens', 'to', 'numerical', 'indices']
['rare', 'words', 'may', 'be', 'replaced', 'with', 'unknown', 'tokens']
['padding', 'allows', 'variable', 'length', 'sentences', 'to', 'form', 'batches']
['subword', 'tokens', 'can', 'represent', 'prefixes', 'roots', 'and', 'suffixes']

<!--Empty Space for separating topics-->


<h2 id="SubwordTokenization">Subword Tokenization</h2>


Subword tokenization breaks words into smaller units. This simple example keeps short words unchanged and splits longer words into character chunks:


In [7]:
# Create a simple subword tokenizer

def subword_tokenize(text, chunk_size=3):
    tokens = []
    for word in word_tokenize(text):
        if len(word) <= chunk_size:
            tokens.append(word)
        else:
            chunks = [word[index:index + chunk_size] for index in range(0, len(word), chunk_size)]
            tokens.extend(chunks)
    return tokens

Compare word-level tokens and subword tokens for one sentence:


In [8]:
# Compare tokenization methods

sample_text = corpus[5]

print("Original text:", sample_text)
print("Word tokens:", word_tokenize(sample_text))
print("Subword tokens:", subword_tokenize(sample_text))

Original text: Subword tokens can represent prefixes, roots, and suffixes.
Word tokens: ['subword', 'tokens', 'can', 'represent', 'prefixes', 'roots', 'and', 'suffixes']
Subword tokens: ['sub', 'wor', 'd', 'tok', 'ens', 'can', 'rep', 'res', 'ent', 'pre', 'fix', 'es', 'roo', 'ts', 'and', 'suf', 'fix', 'es']

Subword tokenization can reduce unknown words because unfamiliar words may still be represented by smaller pieces.


<!--Empty Space for separating topics-->


<h2 id="Vocabulary">Build a Vocabulary with Frequency Filtering</h2>


A vocabulary maps tokens to integer indices. Frequency-based filtering keeps tokens that occur often enough and maps rare tokens to <code>&lt;unk&gt;</code>:


In [9]:
# Count tokens in the corpus

token_counts = Counter(token for tokens in word_tokens for token in tokens)

print(token_counts)

Counter({'tokens': 4, 'text': 2, 'to': 2, 'pytorch': 1, 'models': 1, 'learn': 1, 'from': 1, 'tokenized': 1, 'data': 1, 'tokenization': 1, 'converts': 1, 'raw': 1, 'into': 1, 'useful': 1, 'vocabulary': 1, 'construction': 1, 'maps': 1, 'numerical': 1, 'indices': 1, 'rare': 1, 'words': 1, 'may': 1, 'be': 1, 'replaced': 1, 'with': 1, 'unknown': 1, 'padding': 1, 'allows': 1, 'variable': 1, 'length': 1, 'sentences': 1, 'form': 1, 'batches': 1, 'subword': 1, 'can': 1, 'represent': 1, 'prefixes': 1, 'roots': 1, 'and': 1, 'suffixes': 1})

Build the vocabulary. The special tokens are added first so their indices are stable:


In [10]:
# Build a vocabulary with frequency-based filtering

special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
min_frequency = 2

vocab = {token: index for index, token in enumerate(special_tokens)}

for token, count in token_counts.items():
    if count >= min_frequency and token not in vocab:
        vocab[token] = len(vocab)

print("Vocabulary:", vocab)
print("Vocabulary size:", len(vocab))

Vocabulary: {'<pad>': 0, '<unk>': 1, '<bos>': 2, '<eos>': 3, 'text': 4, 'tokens': 5, 'to': 6}
Vocabulary size: 7

In this small corpus, frequency filtering keeps repeated tokens and treats rare words as unknown.


<!--Empty Space for separating topics-->


<h2 id="Encoding">Encode Tokens as Indices</h2>


Encoding converts tokens into numerical indices that can be stored in PyTorch tensors and passed into neural network layers:


In [11]:
# Encode tokens with special beginning and ending tokens

def encode_tokens(tokens, vocabulary):
    indices = [vocabulary["<bos>"]]
    indices.extend(vocabulary.get(token, vocabulary["<unk>"]) for token in tokens)
    indices.append(vocabulary["<eos>"])
    return indices

Encode all tokenized sentences:


In [12]:
# Convert tokens to indices

encoded_sentences = [encode_tokens(tokens, vocab) for tokens in word_tokens]

for encoded in encoded_sentences:
    print(encoded)

[2, 1, 1, 1, 1, 1, 4, 1, 3]
[2, 1, 1, 1, 4, 1, 1, 5, 3]
[2, 1, 1, 1, 5, 6, 1, 1, 3]
[2, 1, 1, 1, 1, 1, 1, 1, 5, 3]
[2, 1, 1, 1, 1, 1, 6, 1, 1, 3]
[2, 1, 5, 1, 1, 1, 1, 1, 1, 3]

Create one PyTorch tensor from a single encoded sentence:


In [13]:
# Create a tensor from token indices

example_tensor = torch.tensor(encoded_sentences[0], dtype=torch.long)

print(example_tensor)
print(example_tensor.dtype)

tensor([2, 1, 1, 1, 1, 1, 4, 1, 3])
torch.int64

<!--Empty Space for separating topics-->


<h2 id="Decoding">Decode Indices Back to Text</h2>


Decoding converts numerical indices back to tokens. This is useful for inspecting model inputs and outputs:


In [14]:
# Create an index-to-token mapping

index_to_token = {index: token for token, index in vocab.items()}

print(index_to_token)

{0: '<pad>', 1: '<unk>', 2: '<bos>', 3: '<eos>', 4: 'text', 5: 'tokens', 6: 'to'}

Decode one encoded sentence:


In [15]:
# Decode indices to tokens

def decode_indices(indices, reverse_vocabulary):
    return [reverse_vocabulary.get(index, "<unk>") for index in indices]

decoded_tokens = decode_indices(encoded_sentences[0], index_to_token)

print(decoded_tokens)
print(" ".join(decoded_tokens))

['<bos>', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', 'text', '<unk>', '<eos>']
<bos> <unk> <unk> <unk> <unk> <unk> text <unk> <eos>

<!--Empty Space for separating topics-->


<h2 id="SpecialTokens">Special Token Handling</h2>


Special tokens provide structure for model input. <code>&lt;pad&gt;</code> fills short sequences, <code>&lt;unk&gt;</code> handles unseen tokens, <code>&lt;bos&gt;</code> marks the beginning, and <code>&lt;eos&gt;</code> marks the end:


In [16]:
# Store special token indices

pad_index = vocab["<pad>"]
unk_index = vocab["<unk>"]
bos_index = vocab["<bos>"]
eos_index = vocab["<eos>"]

print("PAD:", pad_index)
print("UNK:", unk_index)
print("BOS:", bos_index)
print("EOS:", eos_index)

PAD: 0
UNK: 1
BOS: 2
EOS: 3

Encode a sentence that contains words outside the filtered vocabulary:


In [17]:
# Encode unseen and rare words

new_sentence = "PyTorch handles rare vocabulary items"
new_tokens = word_tokenize(new_sentence)
new_indices = encode_tokens(new_tokens, vocab)

print(new_tokens)
print(new_indices)
print(decode_indices(new_indices, index_to_token))

['pytorch', 'handles', 'rare', 'vocabulary', 'items']
[2, 1, 1, 1, 1, 1, 3]
['<bos>', '<unk>', '<unk>', '<unk>', '<unk>', '<unk>', '<eos>']

<!--Empty Space for separating topics-->


<h2 id="Batching">Padding and Batching</h2>


Text sequences often have different lengths. Use PyTorch padding utilities in a DataLoader collate function to create equal-length batches:


In [18]:
# Create tensors with variable lengths

sequence_tensors = [torch.tensor(sequence, dtype=torch.long) for sequence in encoded_sentences]

for tensor in sequence_tensors:
    print(tensor.shape, tensor)

torch.Size([9]) tensor([2, 1, 1, 1, 1, 1, 4, 1, 3])
torch.Size([9]) tensor([2, 1, 1, 1, 4, 1, 1, 5, 3])
torch.Size([9]) tensor([2, 1, 1, 1, 5, 6, 1, 1, 3])
torch.Size([10]) tensor([2, 1, 1, 1, 1, 1, 1, 1, 5, 3])
torch.Size([10]) tensor([2, 1, 1, 1, 1, 1, 6, 1, 1, 3])
torch.Size([10]) tensor([2, 1, 5, 1, 1, 1, 1, 1, 1, 3])

Create a collate function that pads every batch to the longest sequence in that batch:


In [19]:
# Create a collate function for padding batches

def collate_batch(batch):
    return pad_sequence(batch, batch_first=True, padding_value=pad_index)

Use the collate function with a DataLoader:


In [20]:
# Create a DataLoader with padded batches

loader = DataLoader(sequence_tensors, batch_size=2, shuffle=False, collate_fn=collate_batch)

for batch in loader:
    print(batch)
    print("Batch shape:", batch.shape)
    break

tensor([[2, 1, 1, 1, 1, 1, 4, 1, 3],
        [2, 1, 1, 1, 4, 1, 1, 5, 3]])
Batch shape: torch.Size([2, 9])

The batch tensor has shape <code>[batch_size, padded_sequence_length]</code>. The padded sequence length can change from batch to batch depending on the longest sequence in each batch.



<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


Solve this on your own. Write your code into <code>Lab_2_Tokenization_and_Vocabulary_Construction_using_PyTorch.docx</code>. Create a folder on your local machine and save the file there. Submit your solution at the end of Module 4.


<!--Empty Space for separating topics-->


<h2>Author/s:</h2> 


<b>Harish Pant</b> is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations.




<hr>





## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
